# Préparer le dépôt GitHub + Streamlit
Ce notebook ajoute ton vrai checkpoint au projet, le compacte pour GitHub, valide sa compatibilité puis télécharge le ZIP final.

In [ ]:
# 1) Importe le ZIP reçu de ChatGPT
from google.colab import files
from pathlib import Path
import shutil, subprocess, sys

uploaded = files.upload()
zip_name = next((name for name in uploaded if name.lower().endswith('.zip')), None)
if not zip_name:
    raise RuntimeError('Aucun ZIP importé.')
work = Path('/content/streamlit_build')
if work.exists():
    shutil.rmtree(work)
work.mkdir(parents=True)
shutil.unpack_archive(zip_name, work)
project = work / 'autonomous_driving_demo'
if not (project / 'app.py').is_file():
    candidates = list(work.rglob('app.py'))
    if len(candidates) != 1:
        raise RuntimeError('Impossible de trouver la racine du projet.')
    project = candidates[0].parent
print('Projet :', project)

In [ ]:
# 2) Monte Drive et fabrique le checkpoint compact
from google.colab import drive
drive.mount('/content/drive')

SOURCE_CHECKPOINT = Path('/content/drive/MyDrive/PFE_JEPA/NB6_FINAL_COLAB/detection_E2_architecture/checkpoints/best_detector.pt')
if not SOURCE_CHECKPOINT.is_file():
    raise FileNotFoundError(f'Checkpoint absent : {SOURCE_CHECKPOINT}. Corrige ce chemin dans la cellule.')

destination = project / 'models' / 'best_detector.pt'
subprocess.run([sys.executable, str(project / 'prepare_checkpoint.py'), str(SOURCE_CHECKPOINT), str(destination)], check=True)
parts = sorted(destination.parent.glob(destination.name + '.part*'))
outputs = parts if parts else [destination]
if not all(item.is_file() for item in outputs):
    raise RuntimeError('La conversion du checkpoint a échoué.')
print('Fichiers du modèle :', [item.name for item in outputs])

In [ ]:
# 3) Validation exacte de l'architecture puis création du ZIP final
subprocess.run([sys.executable, str(project / 'validate_deployment.py')], cwd=project, check=True)
final_base = Path('/content/AUTONOMOUS_DRIVING_STREAMLIT_FINAL')
final_zip = Path(shutil.make_archive(str(final_base), 'zip', root_dir=project))
print(f'ZIP final : {final_zip} ({final_zip.stat().st_size / 1024**2:.1f} MiB)')
files.download(str(final_zip))

## Après le téléchargement
Décompresse `AUTONOMOUS_DRIVING_STREAMLIT_FINAL.zip`. Sur GitHub, importe **les fichiers décompressés**, pas le ZIP. Ensuite déploie `app.py` avec Python 3.12 sur Streamlit Community Cloud.